In [2]:
from pathlib import Path
import fnmatch

def print_tree(root,
               max_depth=None,
               show_files=True,
               show_sizes=False,
               human_readable=True,
               ignore_patterns=None):
    """
    打印目录树（类似 `tree` 命令）

    参数
    ----
    root : str | Path
        根目录
    max_depth : int | None
        最大深度（根为深度 0）。None 表示不限制
    show_files : bool
        是否显示文件；False 时只显示文件夹
    show_sizes : bool
        是否显示文件大小
    human_readable : bool
        文件大小是否人类可读（KB/MB/GB）
    ignore_patterns : list[str] | None
        忽略的通配符模式（如 ["*.pyc", "__pycache__", ".git"]）
    """
    root = Path(root)
    if not root.exists():
        print(f"[WARN] path not found: {root}")
        return

    ignore_patterns = ignore_patterns or []

    def _size_str(n):
        if not human_readable:
            return f"{n}B"
        units = ["B", "KB", "MB", "GB", "TB"]
        i = 0
        f = float(n)
        while f >= 1024 and i < len(units) - 1:
            f /= 1024.0
            i += 1
        return f"{f:.1f}{units[i]}"

    def _is_ignored(p: Path) -> bool:
        name = p.name
        for pat in ignore_patterns:
            if fnmatch.fnmatch(name, pat):
                return True
        return False

    def _iter_entries(dir_path: Path):
        # 目录优先，再文件；按名称排序
        dirs, files = [], []
        for p in dir_path.iterdir():
            if _is_ignored(p):
                continue
            (dirs if p.is_dir() else files).append(p)
        dirs.sort(key=lambda x: x.name.lower())
        files.sort(key=lambda x: x.name.lower())
        if show_files:
            return dirs + files
        else:
            return dirs

    def _tree(dir_path: Path, prefix: str = "", depth: int = 0):
        if max_depth is not None and depth > max_depth:
            return
        entries = _iter_entries(dir_path)

        for i, p in enumerate(entries):
            is_last = (i == len(entries) - 1)
            branch = "└── " if is_last else "├── "
            line = prefix + branch + p.name

            if p.is_file() and show_sizes:
                try:
                    size = p.stat().st_size
                    line += f"  ({_size_str(size)})"
                except Exception:
                    line += "  (?)"

            print(line)

            if p.is_dir():
                extension = "    " if is_last else "│   "
                if max_depth is None or depth < max_depth:
                    _tree(p, prefix + extension, depth + 1)

    # 打印根
    print(root.resolve())
    _tree(root, "", 1)


In [3]:
print_tree("t1")

/blue/shiboli.fsu/yifeisun.umich/adversarial_robustness_FNO/1D_Burgers/datasets/1D/Burgers/expanded_pos/t1
└── nu0.0005_pgd
    ├── norm2_eps10_alpha0.01_steps20
    │   └── N=1500
    │       └── dataset.pt
    ├── norm2_eps10_alpha0.05_steps20
    │   └── N=1500
    │       └── dataset.pt
    ├── norm2_eps10_alpha0.1_steps20
    │   └── N=1500
    │       └── dataset.pt
    ├── norm2_eps10_alpha0.5_steps20
    │   └── N=1500
    │       └── dataset.pt
    ├── norm2_eps10_alpha1_steps20
    │   └── N=1500
    │       └── dataset.pt
    ├── norm2_eps20_alpha0.01_steps20
    │   └── N=1500
    │       └── dataset.pt
    ├── norm2_eps20_alpha0.05_steps20
    │   └── N=1500
    │       └── dataset.pt
    ├── norm2_eps20_alpha0.1_steps20
    │   └── N=1500
    │       └── dataset.pt
    ├── norm2_eps20_alpha0.5_steps20
    │   └── N=1500
    │       └── dataset.pt
    ├── norm2_eps20_alpha1_steps20
    │   └── N=1500
    │       └── dataset.pt
    ├── norm2_eps50_alpha0.01_steps20
    │   

In [7]:
import re
from pathlib import Path
import numpy as np
import torch
import matplotlib.pyplot as plt

# =========================
# Utils
# =========================
def _load_pt(pt_path):
    return torch.load(pt_path, map_location="cpu", weights_only=False)

def _to_np(t):
    return t.detach().cpu().numpy() if isinstance(t, torch.Tensor) else np.asarray(t)

def _ensure_1d_sample(arr, idx, t_idx=None):
    """
    从 x 或 y 里取出一条样本的 1D 数据（折线图用）。
    - x 支持 (N, X) 或 (N, X, 1)
    - y 支持 (N, X)、(N, X, 1) 或 (N, X, T)（此时用 t_idx 选择帧）
    返回 1D numpy 向量。
    """
    a = arr[idx]
    if a.ndim == 1:
        return _to_np(a)
    if a.ndim == 2:
        if a.shape[-1] == 1:      # (X,1)
            return _to_np(a[:, 0])
        if t_idx is None:         # (X,T) 需要帧索引
            raise ValueError("y has multiple frames; please provide t_idx")
        tt = t_idx % a.shape[-1]
        return _to_np(a[:, tt])
    raise ValueError(f"Unsupported sample shape: {a.shape} (expect 1D or (X,1) or (X,T))")

def _parse_info_from_name(path_str):
    """
    尽量从文件名里解析关键信息（不强制都能取到）：
    支持：solver, nu, t_final, norm, alpha, epsilon, steps, N, nx
    """
    name = Path(path_str).name
    g = lambda pat: (m.group(1) if (m:=re.search(pat, name)) else "")
    return {
        "solver":  g(r"solver=([A-Za-z0-9_]+)"),
        "nu":      g(r"nu([0-9eE\.\-]+)"),
        "t_final": g(r"t([0-9]+(?:\.[0-9]+)?)"),
        "norm":    g(r"norm([^\W_]+)"),
        "alpha":   g(r"alpha([0-9eE\.\-]+)"),
        "epsilon": g(r"epsilon([0-9eE\.\-]+)"),
        "steps":   g(r"steps([0-9]+)"),
        "N":       g(r"N([0-9]+)"),
        "nx":      g(r"nx([0-9]+)"),
    }

def _short_param_label_for_perturbed(info):
    """
    给 perturbed 的图例用的简短参数标签：norm/eps/alpha/steps 优先。
    """
    parts = []
    if info.get("norm"):    parts.append(f"norm={info['norm']}")
    if info.get("epsilon"): parts.append(f"eps={info['epsilon']}")
    if info.get("alpha"):   parts.append(f"alpha={info['alpha']}")
    if info.get("steps"):   parts.append(f"steps={info['steps']}")
    return ", ".join(parts) if parts else "PGD"

def _wrap_title(text: str, width: int = 80) -> str:
    """
    将每一行按固定宽度硬切：
      - 行长<=width：原样保留
      - 行长>width：按 width 等长切片依次换行
    适用于很长的文件路径/无空格字符串。
    """
    out_lines = []
    for line in text.splitlines():
        if len(line) <= width:
            out_lines.append(line)
        else:
            for i in range(0, len(line), width):
                out_lines.append(line[i:i+width])
    return "\n".join(out_lines)

def _rmse(a, b):
    a = np.asarray(a); b = np.asarray(b)
    return float(np.sqrt(np.mean((a - b) ** 2)))

def _infer_times(T, t_final):
    """
    给 y 的帧生成时间（秒）数组。约定：y 的帧是 1..T，对应 t=Δt,2Δt,...,TΔt。
    x 是 frame 1（t=Δt）。original 的 a 是 frame 0（t=0）。
    """
    if not t_final:
        return None, None
    dt = float(t_final)/float(T)
    t_frames = [(k+1)*dt for k in range(T)]
    t_x = dt
    return t_x, t_frames

def _save_dir_for_pair(attack_pt: Path, original_pt: Path):
    # 把 original 的父目录名带上，避免混淆
    return attack_pt.parent / f"figs_vs_{original_pt.parent.parent.name}_{original_pt.parent.name}"

def _save_name_from_attack_and_idx(attack_pt: Path, idx: int, t_label: str):
    """
    保存名 = attack 的上上级目录_上级目录 + '_idx{idx}_' + t_label + '.png'
    例如：norm2_eps20_alpha0.1_steps20_N=1500_idx7_t19.png
    """
    def two_levels(p: Path):
        a = p.parent.parent.name if p.parent and p.parent.parent else ""
        b = p.parent.name if p.parent else ""
        return f"{a}_{b}" if a or b else p.stem
    base = two_levels(attack_pt)
    return f"{base}_idx{idx}_{t_label}.png"

# =========================
# 单个 attacked .pt 对 original .pt：画若干 index（每个 index 一张图，上下两幅）
# =========================
def plot_compare_one_attacked_vs_original(
    pt_attack: str,
    pt_original: str,
    indices=None,          # 指定索引列表；若为 None 则用前 N 个
    N: int = 5,            # indices 为 None 时取前 N 个
    t_idx: int = -1,       # y 的帧索引（多帧时有效）
    dpi: int = 160,
    figsize=(10.5, 6.2),
    show: bool = False,
    title_wrap: int = 80,
):
    attack_pt = Path(pt_attack)
    orig_pt   = Path(pt_original)

    da = _load_pt(attack_pt)
    do = _load_pt(orig_pt)

    if "x" not in da or "y" not in da or "x" not in do or "y" not in do:
        raise KeyError("Both datasets must contain 'x' and 'y'.")

    xA, yA = da["x"], da["y"]   # perturbed: x=frame1, y=frames 1..T
    xO, yO = do["x"], do["y"]   # original : a=frame0, u=frames 1..T

    # 选择 indices
    if indices is None:
        maxN = min(xA.shape[0], xO.shape[0], yA.shape[0], yO.shape[0])
        useN = min(N, maxN)
        indices = list(range(useN))
    else:
        # 过滤超界
        indices = [i for i in indices if i < xA.shape[0] and i < xO.shape[0] and i < yA.shape[0] and i < yO.shape[0]]
    if not indices:
        print("[warn] no valid indices to plot.")
        return []

    # 帧对齐（y）
    TA = (yA.shape[-1] if (yA.ndim == 3 and yA.shape[-1] > 1) else 1)
    TO = (yO.shape[-1] if (yO.ndim == 3 and yO.shape[-1] > 1) else 1)
    T  = min(TA, TO)
    k  = (t_idx % T) if T > 1 else None

    # 时间标注
    infoA = _parse_info_from_name(attack_pt.name)
    infoO = _parse_info_from_name(orig_pt.name)
    t_final_str = infoA.get("t_final") or infoO.get("t_final") or ""
    t_x, t_frames = _infer_times(T, t_final_str)
    t_k = (t_frames[k] if (t_frames is not None and k is not None) else None)
    t_label = f"t{k+1}" if (k is not None) else "t1"  # 用于文件名的简短帧标

    # 图例标签
    pert_label = f"perturbed ({_short_param_label_for_perturbed(infoA)})"
    orig_label = "original"

    # 保存目录
    save_dir = _save_dir_for_pair(attack_pt, orig_pt)
    save_dir.mkdir(parents=True, exist_ok=True)

    saved = []
    for idx in indices:
        # 取 1D
        x_pert = _ensure_1d_sample(xA, idx, t_idx=None)     # frame1
        x_orig = _ensure_1d_sample(xO, idx, t_idx=None)     # frame0
        y_pert = _ensure_1d_sample(yA, idx, t_idx=k)        # k
        y_orig = _ensure_1d_sample(yO, idx, t_idx=k)        # k

        L = min(len(x_pert), len(x_orig), len(y_pert), len(y_orig))
        x_pert, x_orig = x_pert[:L], x_orig[:L]
        y_pert, y_orig = y_pert[:L], y_orig[:L]
        xs = np.arange(L)

        rmse_x = _rmse(x_pert, x_orig)
        rmse_y = _rmse(y_pert, y_orig)

        fig, axes = plt.subplots(2, 1, figsize=figsize, dpi=dpi, constrained_layout=True)

        # 上：x
        axes[0].plot(xs, x_pert, lw=1.5, label=pert_label)
        axes[0].plot(xs, x_orig, lw=1.5, linestyle="--", label=orig_label)
        sub_title_x = "x (perturbed frame 1) vs a (original frame 0)"
        if t_x is not None:
            sub_title_x += f"   (t={t_x:.3g}s vs 0s)"
        sub_title_x += f"   RMSE={rmse_x:.3e}"
        axes[0].set_title(sub_title_x)
        axes[0].set_xlabel("space index"); axes[0].set_ylabel("value")
        axes[0].legend(loc="best", fontsize=9)
        axes[0].grid(True, alpha=0.3)

        # 下：y
        if T > 1:
            tt = f"frame {k+1}"
            if t_k is not None:
                tt += f" (t={t_k:.3g}s)"
            sub_title_y = f"y ({tt}) : perturbed vs original   RMSE={rmse_y:.3e}"
        else:
            sub_title_y = f"y (single frame) : perturbed vs original   RMSE={rmse_y:.3e}"
        axes[1].plot(xs, y_pert, lw=1.5, label=pert_label)
        axes[1].plot(xs, y_orig, lw=1.5, linestyle="--", label=orig_label)
        # 附加 loss 信息（如存在）
        lossA = da.get("loss", None); lossO = do.get("loss", None)
        tail = []
        if isinstance(lossA, torch.Tensor) and lossA.ndim == 1 and idx < lossA.shape[0]:
            tail.append(f"loss_pert={float(lossA[idx]):.3e}")
        if isinstance(lossO, torch.Tensor) and lossO.ndim == 1 and idx < lossO.shape[0]:
            tail.append(f"loss_orig={float(lossO[idx]):.3e}")
        if tail: sub_title_y += "   " + "  ".join(tail)
        axes[1].set_title(sub_title_y)
        axes[1].set_xlabel("space index"); axes[1].set_ylabel("value")
        axes[1].legend(loc="best", fontsize=9)
        axes[1].grid(True, alpha=0.3)

        # 大标题（两行，含攻防参数 + idx）
        titleA = f"perturbed file: {attack_pt.parent.parent.name}/{attack_pt.parent.name}"
        titleA += f" | {_short_param_label_for_perturbed(infoA)}"
        titleB = f"original file:  {orig_pt.parent.parent.name}/{orig_pt.parent.name}"
        big_title = _wrap_title(f"{titleA}\n{titleB}\nindex={idx}", width=title_wrap)
        fig.suptitle(big_title, y=0.95, fontsize=11)
        fig.subplots_adjust(top=0.8) 

        # 保存
        out_name = _save_name_from_attack_and_idx(attack_pt, idx, t_label)
        out_path = save_dir / out_name
        fig.savefig(out_path, dpi=dpi)
        saved.append(str(out_path))

        if show: plt.show()
        else:    plt.close(fig)

    print(f"[done] {attack_pt.name}: saved {len(saved)} figs -> {save_dir}")
    return saved

# =========================
# 批量：遍历 attacked 目录（递归找 dataset.pt），逐个与 original 对比
# =========================
def batch_plot_folder_vs_original(
    attacked_dir: str,
    original_pt: str,
    indices=None,       # 指定索引列表；None 则取前 N
    N: int = 5,
    t_idx: int = -1,
    pattern: str = "dataset.pt",   # 匹配文件名
    recursive: bool = True,
    **kwargs               # 传给 plot_compare_one_attacked_vs_original 的其它参数
):
    root = Path(attacked_dir)
    if recursive:
        files = sorted(root.rglob(pattern))
    else:
        files = sorted(root.glob(pattern))
    if not files:
        print(f"[warn] no files matched '{pattern}' under: {root}")
        return {}

    all_saved = {}
    for pt_attack in files:
        try:
            saved = plot_compare_one_attacked_vs_original(
                str(pt_attack), original_pt,
                indices=indices, N=N, t_idx=t_idx, **kwargs
            )
            all_saved[str(pt_attack)] = saved
        except Exception as e:
            print(f"[error] {pt_attack}: {e}")
    return all_saved



In [8]:

pt_orig   = "/blue/shiboli.fsu/yifeisun.umich/adversarial_robustness_FNO/1D_Burgers/datasets/1D/Burgers/pos/dim1d_nx1024_N1500_solver=exponax_kernel=gaussian_correlation_length0.03_bcperiodic_nu0.0005_t1.0_seed45.pt"
attacked_dir = "/blue/shiboli.fsu/yifeisun.umich/adversarial_robustness_FNO/1D_Burgers/datasets/1D/Burgers/expanded_pos/t1"
_ = batch_plot_folder_vs_original(
    attacked_dir,
    pt_orig,
    indices=None,    # 不指定就取前 N
    N=5,
    t_idx=-1,
    dpi=160, figsize=(10.5, 6.2), show=False
)

/tmp/ipykernel_997303/1885489438.py:232: UserWarning: This figure was using a layout engine that is incompatible with subplots_adjust and/or tight_layout; not calling subplots_adjust.
  fig.subplots_adjust(top=0.8)


[done] dataset.pt: saved 5 figs -> /blue/shiboli.fsu/yifeisun.umich/adversarial_robustness_FNO/1D_Burgers/datasets/1D/Burgers/expanded_pos/t1/nu0.0005_pgd/norm2_eps10_alpha0.01_steps20/N=1500/figs_vs_Burgers_pos
[done] dataset.pt: saved 5 figs -> /blue/shiboli.fsu/yifeisun.umich/adversarial_robustness_FNO/1D_Burgers/datasets/1D/Burgers/expanded_pos/t1/nu0.0005_pgd/norm2_eps10_alpha0.05_steps20/N=1500/figs_vs_Burgers_pos
[done] dataset.pt: saved 5 figs -> /blue/shiboli.fsu/yifeisun.umich/adversarial_robustness_FNO/1D_Burgers/datasets/1D/Burgers/expanded_pos/t1/nu0.0005_pgd/norm2_eps10_alpha0.1_steps20/N=1500/figs_vs_Burgers_pos
[done] dataset.pt: saved 5 figs -> /blue/shiboli.fsu/yifeisun.umich/adversarial_robustness_FNO/1D_Burgers/datasets/1D/Burgers/expanded_pos/t1/nu0.0005_pgd/norm2_eps10_alpha0.5_steps20/N=1500/figs_vs_Burgers_pos
[done] dataset.pt: saved 5 figs -> /blue/shiboli.fsu/yifeisun.umich/adversarial_robustness_FNO/1D_Burgers/datasets/1D/Burgers/expanded_pos/t1/nu0.0005_pgd

In [5]:
base_dir = "/blue/shiboli.fsu/yifeisun.umich/adversarial_robustness_FNO/1D_Burgers/datasets/1D/Burgers/expanded_pos/t1"

loop_plot_firstN_stacked(
    base_dir,
    N=5,
    t_idx=-1,
    save_root="./quick_view_xy_pairs",  # 统一保存到这个根目录，按数据集子目录分层
    show=False,
)


[process] /blue/shiboli.fsu/yifeisun.umich/adversarial_robustness_FNO/1D_Burgers/datasets/1D/Burgers/expanded_pos/t1/nu0.0005_pgd/norm2_eps10_alpha0.01_steps20/N=1500/dataset.pt
[done] /blue/shiboli.fsu/yifeisun.umich/adversarial_robustness_FNO/1D_Burgers/datasets/1D/Burgers/expanded_pos/t1/nu0.0005_pgd/norm2_eps10_alpha0.01_steps20/N=1500/dataset.pt  -> saved 5 figures to: quick_view_xy_pairs/norm2_eps10_alpha0.01_steps20/N=1500
[process] /blue/shiboli.fsu/yifeisun.umich/adversarial_robustness_FNO/1D_Burgers/datasets/1D/Burgers/expanded_pos/t1/nu0.0005_pgd/norm2_eps10_alpha0.05_steps20/N=1500/dataset.pt
[done] /blue/shiboli.fsu/yifeisun.umich/adversarial_robustness_FNO/1D_Burgers/datasets/1D/Burgers/expanded_pos/t1/nu0.0005_pgd/norm2_eps10_alpha0.05_steps20/N=1500/dataset.pt  -> saved 5 figures to: quick_view_xy_pairs/norm2_eps10_alpha0.05_steps20/N=1500
[process] /blue/shiboli.fsu/yifeisun.umich/adversarial_robustness_FNO/1D_Burgers/datasets/1D/Burgers/expanded_pos/t1/nu0.0005_pgd/no

{'/blue/shiboli.fsu/yifeisun.umich/adversarial_robustness_FNO/1D_Burgers/datasets/1D/Burgers/expanded_pos/t1/nu0.0005_pgd/norm2_eps10_alpha0.01_steps20/N=1500/dataset.pt': ['quick_view_xy_pairs/norm2_eps10_alpha0.01_steps20/N=1500/norm2_eps10_alpha0.01_steps20_N=1500_idx0.png',
  'quick_view_xy_pairs/norm2_eps10_alpha0.01_steps20/N=1500/norm2_eps10_alpha0.01_steps20_N=1500_idx1.png',
  'quick_view_xy_pairs/norm2_eps10_alpha0.01_steps20/N=1500/norm2_eps10_alpha0.01_steps20_N=1500_idx2.png',
  'quick_view_xy_pairs/norm2_eps10_alpha0.01_steps20/N=1500/norm2_eps10_alpha0.01_steps20_N=1500_idx3.png',
  'quick_view_xy_pairs/norm2_eps10_alpha0.01_steps20/N=1500/norm2_eps10_alpha0.01_steps20_N=1500_idx4.png'],
 '/blue/shiboli.fsu/yifeisun.umich/adversarial_robustness_FNO/1D_Burgers/datasets/1D/Burgers/expanded_pos/t1/nu0.0005_pgd/norm2_eps10_alpha0.05_steps20/N=1500/dataset.pt': ['quick_view_xy_pairs/norm2_eps10_alpha0.05_steps20/N=1500/norm2_eps10_alpha0.05_steps20_N=1500_idx0.png',
  'quick_

In [2]:
import os
import re
from pathlib import Path
from typing import List, Dict, Tuple, Optional

import torch
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from matplotlib.colors import to_hex

# ===================== 基础工具 =====================

def _load_pt(pt_path: str) -> Dict[str, torch.Tensor]:
    data = torch.load(pt_path, map_location="cpu", weights_only=False)
    if "x" not in data or "y" not in data:
        raise KeyError(f"{pt_path} missing 'x' or 'y'. Keys={list(data.keys())}")
    return data

def _to_1d_line(t: torch.Tensor, idx: int, t_idx: Optional[int]) -> np.ndarray:
    """
    取出某个样本的 1D 曲线（支持 (N,X), (N,X,1), (N,X,T)）。
    - 对 x：忽略 t_idx，只需 (N,X) 或 (N,X,1)
    - 对 y：若是 (N,X,T)，按 t_idx 取帧
    """
    arr = t[idx]
    if arr.ndim == 2 and arr.shape[-1] == 1:   # (X,1) -> (X,)
        arr = arr[..., 0]
    if arr.ndim == 2 and t_idx is not None:    # (X,T) -> (X,)
        T = arr.shape[-1]
        k = t_idx % T
        arr = arr[..., k]
    if arr.ndim != 1:
        raise ValueError(f"Expect 1D line after selection, got shape {tuple(arr.shape)}")
    return arr.detach().cpu().numpy()

def _rmse(a: np.ndarray, b: np.ndarray) -> float:
    a = np.asarray(a); b = np.asarray(b)
    return float(np.sqrt(np.mean((a - b) ** 2)))

def _parse_meta_from_path(p: str) -> Dict[str, str]:
    """从路径/文件名里尽量解析关键信息，解析不到的留空。"""
    s = str(p)
    meta = {}
    # 常见攻击超参出现在中间目录或文件名里：
    m = re.search(r"norm([A-Za-z0-9∞]+)", s);       meta["norm"]  = m.group(1) if m else ""
    m = re.search(r"eps(?:ilon)?([0-9eE\.\-]+)", s);meta["epsilon"] = m.group(1) if m else ""
    m = re.search(r"alpha([0-9eE\.\-]+)", s);       meta["alpha"] = m.group(1) if m else ""
    m = re.search(r"steps([0-9]+)", s);            meta["steps"] = m.group(1) if m else ""
    m = re.search(r"solver=([A-Za-z0-9_]+)", s);   meta["solver"] = m.group(1) if m else ""
    m = re.search(r"nu([0-9eE\.\-]+)", s);         meta["nu"]     = m.group(1) if m else ""
    m = re.search(r"t([0-9]+(?:\.[0-9]+)?)", s);   meta["t_final"]= m.group(1) if m else ""
    return meta

def _walk_attack_datasets(root_dir: str) -> List[str]:
    """在 root_dir 下递归寻找所有名为 dataset.pt 的攻击数据集路径。"""
    hits = []
    for r, _, files in os.walk(root_dir):
        for f in files:
            if f == "dataset.pt":
                hits.append(os.path.join(r, f))
    if not hits:
        raise FileNotFoundError(f"No dataset.pt found under {root_dir}")
    return sorted(hits)

def _get_float(s: str) -> Optional[float]:
    try:
        return float(s)
    except Exception:
        return None

def _rank_colors(values: List[float], cmap_name: str = "viridis") -> Dict[float, str]:
    """
    按值排序映射到等间隔 colormap（用 rank，而不是用绝对数值）。
    返回 {value: '#RRGGBB'}
    """
    uniq = sorted(set(values))
    if len(uniq) == 1:
        return {uniq[0]: to_hex(cm.get_cmap(cmap_name)(0.6))}
    cmap = cm.get_cmap(cmap_name)
    mapping = {}
    for i, v in enumerate(uniq):
        t = i / (len(uniq) - 1)  # 0..1
        mapping[v] = to_hex(cmap(t))
    return mapping

def _wrap_title(txt: str, width: int = 80) -> str:
    """简单按空格断行，避免标题过长挤压。"""
    words = txt.split()
    lines, cur, cur_len = [], [], 0
    for w in words:
        if cur_len + len(w) + (1 if cur else 0) > width:
            lines.append(" ".join(cur))
            cur, cur_len = [w], len(w)
        else:
            cur.append(w); cur_len += len(w) + (1 if cur_len else 0)
    if cur: lines.append(" ".join(cur))
    return "\n".join(lines)

# ===================== 绘图（固定 ε：扫所有 α） =====================

def plot_index_by_epsilon(
    attacks_root: str,
    pt_original: str,
    idx: int,
    epsilon_select: float,
    t_idx: Optional[int] = None,      # y/u 的帧号（None 表示 y、u 都是单帧或不用取帧）
    figsize_per_row: float = 3.0,
    x_label: str = "spatial index",
    suptitle_wrap: int = 120,
    subplots_top: float = 0.86,       # 控制标题与图的间距
):
    """
    从 attacks_root 下搜集所有 dataset.pt，筛选出 ε==epsilon_select 的数据集，
    将不同 α 的曲线（x 与 y）与 original 同图对比。颜色按 α 的排序名次赋色。
    """
    # 找到所有攻击数据集
    pt_list = _walk_attack_datasets(attacks_root)
    # 载入 original（a, u）
    d0 = _load_pt(pt_original)
    a_line = _to_1d_line(d0["x"], idx, t_idx=None)         # original 的 a（x）
    u_line = _to_1d_line(d0["y"], idx, t_idx=t_idx)        # original 的 u（y 同帧）

    # 过滤出指定 epsilon 的攻击数据集
    chosen: List[Tuple[str, Dict[str, str]]] = []
    for p in pt_list:
        meta = _parse_meta_from_path(p)
        eps = _get_float(meta.get("epsilon", ""))
        if eps is None:
            continue
        if abs(eps - float(epsilon_select)) < 1e-12:
            chosen.append((p, meta))
    if not chosen:
        raise ValueError(f"No attack datasets with epsilon={epsilon_select} under {attacks_root}")

    # 收集所有 alpha 值并确定颜色（按 rank）
    alpha_vals = []
    for _, meta in chosen:
        a = _get_float(meta.get("alpha", ""))
        if a is not None:
            alpha_vals.append(a)
    if not alpha_vals:
        raise ValueError("No alpha parsed from selected datasets.")
    color_map = _rank_colors(alpha_vals, cmap_name="viridis")

    # 画图：一张图，两行（上：x，对比 a；下：y，对比 u）
    fig, axes = plt.subplots(
        2, 1, figsize=(8 * figsize_per_row / 3, 6.5), sharex=True, constrained_layout=False
    )
    ax_x, ax_y = axes

    # Original 基准线
    ax_x.plot(a_line, color="k", lw=2.0, label="original (RMSE=0)")
    ax_y.plot(u_line, color="k", lw=2.0, label="original (RMSE=0)")

    # 各 α 的曲线
    legends_x = []
    legends_y = []
    for p, meta in sorted(chosen, key=lambda t: _get_float(t[1].get("alpha","")) or 0.0):
        da = _load_pt(p)

        x_line = _to_1d_line(da["x"], idx, t_idx=None)
        y_line = _to_1d_line(da["y"], idx, t_idx=t_idx)

        rmse_x = _rmse(x_line, a_line)
        rmse_y = _rmse(y_line, u_line)

        alpha_val = _get_float(meta.get("alpha",""))
        color = color_map.get(alpha_val, None) if alpha_val is not None else None

        lbl_x = f"perturbed α={alpha_val:g} (RMSE vs orig={rmse_x:.3e})"
        lbl_y = f"perturbed α={alpha_val:g} (RMSE vs orig={rmse_y:.3e})"

        ax_x.plot(x_line, label=lbl_x, color=color, alpha=0.95)
        ax_y.plot(y_line, label=lbl_y, color=color, alpha=0.95)
        legends_x.append(lbl_x); legends_y.append(lbl_y)

    # 轴与图例
    ax_y.set_xlabel(x_label)
    ax_x.set_ylabel("x / a")
    ax_y.set_ylabel("y / u")
    ax_x.legend(fontsize=8, ncol=1, loc="upper right")
    ax_y.legend(fontsize=8, ncol=1, loc="upper right")
    ax_x.grid(True, alpha=0.2); ax_y.grid(True, alpha=0.2)

    # 大标题（换行 & 解析信息）
    meta0 = _parse_meta_from_path(chosen[0][0])
    desc = f"solver={meta0.get('solver','')}, nu={meta0.get('nu','')}, norm={meta0.get('norm','')}, steps={meta0.get('steps','')}, T={meta0.get('t_final','')}"
    title = f"[Fixed ε={epsilon_select:g}] Varying α — index={idx}\n{desc}\nroot={attacks_root}\norig={pt_original}"
    fig.suptitle(_wrap_title(title, width=suptitle_wrap), fontsize=11)
    fig.subplots_adjust(top=subplots_top)

    return fig, axes


# ===================== 绘图（固定 α：扫所有 ε） =====================

def plot_index_by_alpha(
    attacks_root: str,
    pt_original: str,
    idx: int,
    alpha_select: float,
    t_idx: Optional[int] = None,
    figsize_per_row: float = 3.0,
    x_label: str = "spatial index",
    suptitle_wrap: int = 120,
    subplots_top: float = 0.86,
):
    """
    从 attacks_root 下搜集所有 dataset.pt，筛选出 α==alpha_select 的数据集，
    将不同 ε 的曲线与 original 同图对比。颜色按 ε 的排序名次赋色。
    """
    pt_list = _walk_attack_datasets(attacks_root)
    d0 = _load_pt(pt_original)
    a_line = _to_1d_line(d0["x"], idx, t_idx=None)
    u_line = _to_1d_line(d0["y"], idx, t_idx=t_idx)

    chosen: List[Tuple[str, Dict[str, str]]] = []
    for p in pt_list:
        meta = _parse_meta_from_path(p)
        alpha_val = _get_float(meta.get("alpha",""))
        if alpha_val is None:
            continue
        if abs(alpha_val - float(alpha_select)) < 1e-12:
            chosen.append((p, meta))
    if not chosen:
        raise ValueError(f"No attack datasets with alpha={alpha_select} under {attacks_root}")

    eps_vals = []
    for _, meta in chosen:
        ev = _get_float(meta.get("epsilon",""))
        if ev is not None: eps_vals.append(ev)
    if not eps_vals:
        raise ValueError("No epsilon parsed from selected datasets.")
    color_map = _rank_colors(eps_vals, cmap_name="plasma")

    fig, axes = plt.subplots(
        2, 1, figsize=(8 * figsize_per_row / 3, 6.5), sharex=True, constrained_layout=False
    )
    ax_x, ax_y = axes
    ax_x.plot(a_line, color="k", lw=2.0, label="original (RMSE=0)")
    ax_y.plot(u_line, color="k", lw=2.0, label="original (RMSE=0)")

    for p, meta in sorted(chosen, key=lambda t: _get_float(t[1].get("epsilon","")) or 0.0):
        da = _load_pt(p)
        x_line = _to_1d_line(da["x"], idx, t_idx=None)
        y_line = _to_1d_line(da["y"], idx, t_idx=t_idx)

        rmse_x = _rmse(x_line, a_line)
        rmse_y = _rmse(y_line, u_line)

        eps_val = _get_float(meta.get("epsilon",""))
        color = color_map.get(eps_val, None) if eps_val is not None else None

        lbl_x = f"perturbed ε={eps_val:g} (RMSE vs orig={rmse_x:.3e})"
        lbl_y = f"perturbed ε={eps_val:g} (RMSE vs orig={rmse_y:.3e})"

        ax_x.plot(x_line, label=lbl_x, color=color, alpha=0.95)
        ax_y.plot(y_line, label=lbl_y, color=color, alpha=0.95)

    ax_y.set_xlabel(x_label)
    ax_x.set_ylabel("x / a")
    ax_y.set_ylabel("y / u")
    ax_x.legend(fontsize=8, ncol=1, loc="upper right")
    ax_y.legend(fontsize=8, ncol=1, loc="upper right")
    ax_x.grid(True, alpha=0.2); ax_y.grid(True, alpha=0.2)

    meta0 = _parse_meta_from_path(chosen[0][0])
    desc = f"solver={meta0.get('solver','')}, nu={meta0.get('nu','')}, norm={meta0.get('norm','')}, steps={meta0.get('steps','')}, T={meta0.get('t_final','')}"
    title = f"[Fixed α={alpha_select:g}] Varying ε — index={idx}\n{desc}\nroot={attacks_root}\norig={pt_original}"
    fig.suptitle(_wrap_title(title, width=suptitle_wrap), fontsize=11)
    fig.subplots_adjust(top=subplots_top)

    return fig, axes

# ===================== 用法示例 =====================
# attacks_root = "/blue/.../datasets/1D/Burgers/expanded_pos/t1/nu0.0005_pgd"
# pt_orig      = "/blue/.../datasets/1D/Burgers/pos/dim1d_nx1024_N1500_...nu0.0005_t1.0_seed45.pt"
# idx = 69
# fig, axes = plot_index_by_epsilon(attacks_root, pt_orig, idx, epsilon_select=20, t_idx=None)
# fig, axes = plot_index_by_alpha(attacks_root, pt_orig, idx, alpha_select=0.1, t_idx=None)
# plt.show()
# === 批处理：把所有 epsilon/alpha 和若干 idx 都画出来 ===
def plot_all_epsilon_alpha_indices(
    attacks_root: str,
    pt_original: str,
    idx_list=(0,1,2,3,4),
    save_dir="./plots_batch",
    t_idx=None
):
    os.makedirs(save_dir, exist_ok=True)

    # 先解析所有攻击数据集的 meta
    pt_list = _walk_attack_datasets(attacks_root)
    eps_vals, alpha_vals = set(), set()
    for p in pt_list:
        meta = _parse_meta_from_path(p)
        ev = _get_float(meta.get("epsilon",""))
        av = _get_float(meta.get("alpha",""))
        if ev is not None: eps_vals.add(ev)
        if av is not None: alpha_vals.add(av)

    eps_sorted = sorted(eps_vals)
    alpha_sorted = sorted(alpha_vals)

    print(f"Found epsilons={eps_sorted}")
    print(f"Found alphas={alpha_sorted}")

    # 循环 epsilon, alpha, idx
    for eps in eps_sorted:
        for idx in idx_list:
            fig, axes = plot_index_by_epsilon(
                attacks_root, pt_original, idx,
                epsilon_select=eps, t_idx=t_idx
            )
            out_path = os.path.join(
                save_dir, f"byEps_eps{eps}_idx{idx}.png"
            )
            fig.savefig(out_path, dpi=150)
            plt.close(fig)
            print(f"Saved {out_path}")

    for alpha in alpha_sorted:
        for idx in idx_list:
            fig, axes = plot_index_by_alpha(
                attacks_root, pt_original, idx,
                alpha_select=alpha, t_idx=t_idx
            )
            out_path = os.path.join(
                save_dir, f"byAlpha_alpha{alpha}_idx{idx}.png"
            )
            fig.savefig(out_path, dpi=150)
            plt.close(fig)
            print(f"Saved {out_path}")


In [3]:
# ======= 新增：均值辅助 =======
def _mean_1d_x(data: Dict[str, torch.Tensor]) -> np.ndarray:
    """对 x 在样本维度做平均 -> (X,)；兼容 (N,X) 或 (N,X,1)。"""
    x = data["x"]
    if x.ndim == 3 and x.shape[-1] == 1:
        x = x[..., 0]
    if x.ndim != 2:
        raise ValueError(f"x must be (N,X) or (N,X,1); got {tuple(x.shape)}")
    return x.mean(dim=0).detach().cpu().numpy()

def _mean_1d_y(data: Dict[str, torch.Tensor], t_idx: Optional[int]) -> np.ndarray:
    """
    对 y 在样本维度做平均 -> (X,)。
    兼容 (N,X) 或 (N,X,T)。若 (N,X,T)，按 t_idx 取帧后再均值；t_idx=None 时取最后一帧。
    """
    y = data["y"]
    if y.ndim == 3:  # (N, X, T) 或 (N, X)（1D 情况这里一般是 (N,X)）
        if y.shape[-1] > 1:  # 说明是 (N, X, T)
            T = y.shape[-1]
            k = (t_idx % T) if (t_idx is not None) else (T - 1)
            y = y[..., k]
        # 现在是 (N, X)
    elif y.ndim != 2:
        raise ValueError(f"y must be (N,X) or (N,X,T); got {tuple(y.shape)}")
    return y.mean(dim=0).detach().cpu().numpy()

# ======= 新增：单图（均值版）- 固定 ε，扫 α =======
def plot_mean_by_epsilon(
    attacks_root: str,
    pt_original: str,
    epsilon_select: float,
    t_idx: Optional[int] = None,
    figsize_per_row: float = 3.0,
    x_label: str = "spatial index",
    suptitle_wrap: int = 120,
    subplots_top: float = 0.86,
):
    pt_list = _walk_attack_datasets(attacks_root)

    # 原始均值曲线
    d0 = _load_pt(pt_original)
    a_mean = _mean_1d_x(d0)               # original 的 a
    u_mean = _mean_1d_y(d0, t_idx=t_idx)  # original 的 u(同帧)

    # 过滤 epsilon
    chosen = []
    for p in pt_list:
        meta = _parse_meta_from_path(p)
        eps = _get_float(meta.get("epsilon", ""))
        if eps is None:
            continue
        if abs(eps - float(epsilon_select)) < 1e-12:
            chosen.append((p, meta))
    if not chosen:
        raise ValueError(f"No attack datasets with epsilon={epsilon_select} under {attacks_root}")

    # α 的 rank 映射颜色
    alpha_vals = []
    for _, meta in chosen:
        av = _get_float(meta.get("alpha", ""))
        if av is not None:
            alpha_vals.append(av)
    if not alpha_vals:
        raise ValueError("No alpha parsed from selected datasets.")
    color_map = _rank_colors(alpha_vals, cmap_name="viridis")

    # 画两行：上 x-mean vs a-mean，下 y-mean vs u-mean
    fig, axes = plt.subplots(2, 1, figsize=(8 * figsize_per_row / 3, 6.5), sharex=True, constrained_layout=False)
    ax_x, ax_y = axes

    ax_x.plot(a_mean, color="k", lw=2.0, label="original mean")
    ax_y.plot(u_mean, color="k", lw=2.0, label="original mean")

    for p, meta in sorted(chosen, key=lambda t: _get_float(t[1].get("alpha", "")) or 0.0):
        da = _load_pt(p)
        x_mean = _mean_1d_x(da)
        y_mean = _mean_1d_y(da, t_idx=t_idx)

        rmse_x = _rmse(x_mean, a_mean)
        rmse_y = _rmse(y_mean, u_mean)

        alpha_val = _get_float(meta.get("alpha", ""))
        color = color_map.get(alpha_val) if alpha_val is not None else None

        ax_x.plot(x_mean, label=f"perturbed α={alpha_val:g} (RMSE={rmse_x:.3e})", color=color, alpha=0.95)
        ax_y.plot(y_mean, label=f"perturbed α={alpha_val:g} (RMSE={rmse_y:.3e})", color=color, alpha=0.95)

    ax_y.set_xlabel(x_label)
    ax_x.set_ylabel("x (mean) / a (mean)")
    ax_y.set_ylabel("y (mean) / u (mean)")
    ax_x.legend(fontsize=8, loc="upper right")
    ax_y.legend(fontsize=8, loc="upper right")
    ax_x.grid(True, alpha=0.2); ax_y.grid(True, alpha=0.2)

    meta0 = _parse_meta_from_path(chosen[0][0])
    desc = f"solver={meta0.get('solver','')}, nu={meta0.get('nu','')}, norm={meta0.get('norm','')}, steps={meta0.get('steps','')}, T={meta0.get('t_final','')}"
    title = f"[MEAN | Fixed ε={epsilon_select:g}] Varying α\n{desc}\nroot={attacks_root}\norig={pt_original}"
    fig.suptitle(_wrap_title(title, width=suptitle_wrap), fontsize=11)
    fig.subplots_adjust(top=subplots_top)
    return fig, axes

# ======= 新增：单图（均值版）- 固定 α，扫 ε =======
def plot_mean_by_alpha(
    attacks_root: str,
    pt_original: str,
    alpha_select: float,
    t_idx: Optional[int] = None,
    figsize_per_row: float = 3.0,
    x_label: str = "spatial index",
    suptitle_wrap: int = 120,
    subplots_top: float = 0.86,
):
    pt_list = _walk_attack_datasets(attacks_root)

    d0 = _load_pt(pt_original)
    a_mean = _mean_1d_x(d0)
    u_mean = _mean_1d_y(d0, t_idx=t_idx)

    chosen = []
    for p in pt_list:
        meta = _parse_meta_from_path(p)
        av = _get_float(meta.get("alpha", ""))
        if av is None:
            continue
        if abs(av - float(alpha_select)) < 1e-12:
            chosen.append((p, meta))
    if not chosen:
        raise ValueError(f"No attack datasets with alpha={alpha_select} under {attacks_root}")

    eps_vals = []
    for _, meta in chosen:
        ev = _get_float(meta.get("epsilon", ""))
        if ev is not None: eps_vals.append(ev)
    if not eps_vals:
        raise ValueError("No epsilon parsed from selected datasets.")
    color_map = _rank_colors(eps_vals, cmap_name="plasma")

    fig, axes = plt.subplots(2, 1, figsize=(8 * figsize_per_row / 3, 6.5), sharex=True, constrained_layout=False)
    ax_x, ax_y = axes

    ax_x.plot(a_mean, color="k", lw=2.0, label="original mean")
    ax_y.plot(u_mean, color="k", lw=2.0, label="original mean")

    for p, meta in sorted(chosen, key=lambda t: _get_float(t[1].get("epsilon","")) or 0.0):
        da = _load_pt(p)
        x_mean = _mean_1d_x(da)
        y_mean = _mean_1d_y(da, t_idx=t_idx)

        rmse_x = _rmse(x_mean, a_mean)
        rmse_y = _rmse(y_mean, u_mean)

        eps_val = _get_float(meta.get("epsilon",""))
        color = color_map.get(eps_val) if eps_val is not None else None

        ax_x.plot(x_mean, label=f"perturbed ε={eps_val:g} (RMSE={rmse_x:.3e})", color=color, alpha=0.95)
        ax_y.plot(y_mean, label=f"perturbed ε={eps_val:g} (RMSE={rmse_y:.3e})", color=color, alpha=0.95)

    ax_y.set_xlabel(x_label)
    ax_x.set_ylabel("x (mean) / a (mean)")
    ax_y.set_ylabel("y (mean) / u (mean)")
    ax_x.legend(fontsize=8, loc="upper right")
    ax_y.legend(fontsize=8, loc="upper right")
    ax_x.grid(True, alpha=0.2); ax_y.grid(True, alpha=0.2)

    meta0 = _parse_meta_from_path(chosen[0][0])
    desc = f"solver={meta0.get('solver','')}, nu={meta0.get('nu','')}, norm={meta0.get('norm','')}, steps={meta0.get('steps','')}, T={meta0.get('t_final','')}"
    title = f"[MEAN | Fixed α={alpha_select:g}] Varying ε\n{desc}\nroot={attacks_root}\norig={pt_original}"
    fig.suptitle(_wrap_title(title, width=suptitle_wrap), fontsize=11)
    fig.subplots_adjust(top=subplots_top)
    return fig, axes

# ======= 替换：批量函数改为“对所有 index 求平均” =======
def plot_all_epsilon_alpha_indices(   # ← 名称保持不变，但现在做均值
    attacks_root: str,
    pt_original: str,
    save_dir="./plots_batch_mean",
    t_idx=None
):
    """
    批量输出“均值图”：对所有样本求平均后作图。
    1) 固定 ε，扫 α -> 保存 byEps_eps{eps}_MEAN.png
    2) 固定 α，扫 ε -> 保存 byAlpha_alpha{alpha}_MEAN.png
    """
    os.makedirs(save_dir, exist_ok=True)

    pt_list = _walk_attack_datasets(attacks_root)
    eps_vals, alpha_vals = set(), set()
    for p in pt_list:
        meta = _parse_meta_from_path(p)
        ev = _get_float(meta.get("epsilon",""))
        av = _get_float(meta.get("alpha",""))
        if ev is not None: eps_vals.add(ev)
        if av is not None: alpha_vals.add(av)

    eps_sorted = sorted(eps_vals)
    alpha_sorted = sorted(alpha_vals)

    print(f"[MEAN] Found epsilons={eps_sorted}")
    print(f"[MEAN] Found alphas={alpha_sorted}")

    for eps in eps_sorted:
        fig, _ = plot_mean_by_epsilon(attacks_root, pt_original, epsilon_select=eps, t_idx=t_idx)
        out_path = os.path.join(save_dir, f"byEps_eps{eps}_MEAN.png")
        fig.savefig(out_path, dpi=160, bbox_inches="tight")
        plt.close(fig)
        print(f"Saved {out_path}")

    for alpha in alpha_sorted:
        fig, _ = plot_mean_by_alpha(attacks_root, pt_original, alpha_select=alpha, t_idx=t_idx)
        out_path = os.path.join(save_dir, f"byAlpha_alpha{alpha}_MEAN.png")
        fig.savefig(out_path, dpi=160, bbox_inches="tight")
        plt.close(fig)
        print(f"Saved {out_path}")


In [5]:
pt_orig = "/blue/shiboli.fsu/yifeisun.umich/adversarial_robustness_FNO/1D_Burgers/datasets/1D/Burgers/pos/dim1d_nx1024_N1500_solver=exponax_kernel=gaussian_correlation_length0.03_bcperiodic_nu0.0005_t1.0_seed45.pt"
attacks_root = "/blue/shiboli.fsu/yifeisun.umich/adversarial_robustness_FNO/1D_Burgers/datasets/1D/Burgers/expanded_pos/t1"

plot_all_epsilon_alpha_indices(
    attacks_root,
    pt_orig,
    save_dir="./plots_batch",
    t_idx=None   # 如果 y 是多帧，可以改成 -1 或具体帧号
)


[MEAN] Found epsilons=[10.0, 20.0, 50.0]
[MEAN] Found alphas=[0.01, 0.05, 0.1, 0.5, 1.0]


/tmp/ipykernel_997303/2291167406.py:80: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  cmap = cm.get_cmap(cmap_name)


Saved ./plots_batch/byEps_eps10.0_MEAN.png
Saved ./plots_batch/byEps_eps20.0_MEAN.png
Saved ./plots_batch/byEps_eps50.0_MEAN.png
Saved ./plots_batch/byAlpha_alpha0.01_MEAN.png
Saved ./plots_batch/byAlpha_alpha0.05_MEAN.png
Saved ./plots_batch/byAlpha_alpha0.1_MEAN.png
Saved ./plots_batch/byAlpha_alpha0.5_MEAN.png
Saved ./plots_batch/byAlpha_alpha1.0_MEAN.png
